# Fashion Retriever — Natural Language Search

This notebook loads the artifacts built by the **The_Indexer notebook**:
- `inventory.csv`
- `fashion_metadata_1500.json` (Florence captions + spaCy structured tags)
- `clip_embeddings.npy`
- `faiss.index`

and exposes a single function, `search(query, k=9)`, that returns the top-k matching images for a natural language query.

## Section 1 — Setup & Load Artifacts

In [ ]:
!pip install -q open_clip_torch faiss-gpu spacy
!python -m spacy download en_core_web_sm

In [ ]:
import json

import torch
import numpy as np
import pandas as pd
import faiss
import open_clip
import spacy
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# ---------------------------------------------------------
# Paths to the artifacts produced by the indexer notebook.
# If this retriever runs in the SAME Kaggle session as the
# indexer, these live under /kaggle/working/.
# If you're loading them as a separate Kaggle input dataset,
# change ARTIFACT_DIR to that input path instead.
# ---------------------------------------------------------

ARTIFACT_DIR = Path("/kaggle/input/datasets/riddhipandya1/results")

INVENTORY_PATH = ARTIFACT_DIR / "inventory.csv"
METADATA_PATH  = ARTIFACT_DIR / "fashion_metadata_1500.json"
FAISS_PATH     = ARTIFACT_DIR / "faiss.index"

inventory = pd.read_csv(INVENTORY_PATH)

with open(METADATA_PATH, "r") as f:
    metadata_records = json.load(f)

metadata_df = pd.DataFrame(metadata_records)

faiss_index = faiss.read_index(str(FAISS_PATH))

print("Inventory rows:", len(inventory))
print("Metadata rows:", len(metadata_df))
print("FAISS vectors:", faiss_index.ntotal)

In [ ]:
# The FAISS index was built by iterating over metadata_df's rows
# in order (see indexer Section 7), so FAISS row i <-> metadata_df row i.
# This lets us go from a FAISS search result straight to full metadata.

assert faiss_index.ntotal == len(metadata_df), (
    "FAISS index size and metadata row count don't match — "
    "make sure both come from the same indexer run."
)

## Section 2 — Load CLIP (for encoding the query text)

In [ ]:
weights_path = "/kaggle/input/datasets/riddhipandya1/clip-tensors-neww/open_clip_model.safetensors"

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained=weights_path
)

clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")

clip_model = clip_model.to(device)
clip_model.eval()

print("CLIP loaded")

## Section 3 — Query Understanding (spaCy)

Same tagger used at index time, reused here to parse the **query string** into
structured attributes (color, garment, footwear, setting, etc.). This is what
gives the retriever context-awareness for multi-attribute queries.

In [ ]:
nlp = spacy.load("en_core_web_sm")
print("spaCy loaded")

In [ ]:
def extract_spacy_features(caption):

    text = caption.lower()
    doc = nlp(text)

    # --------------------------------------------------
    # Vocabulary
    # NOTE: sets store BASE (lemma) forms. Matching checks both a
    # token's lemma and its raw text, so inflected forms like
    # "standing" / "stood" / "stands" or "sat" / "sitting" all match
    # the same base entry ("stand", "sit"). This fixes the earlier
    # bug where -ing-only vocab entries missed most real captions.
    # --------------------------------------------------

    COLORS = {
        "black","white","red","blue","green","yellow","orange","pink",
        "purple","brown","grey","gray","beige","cream","gold","silver",
        "maroon","navy","olive","teal","cyan","turquoise"
    }

    NECKLINES = {
        "crew neck","crew-neck","round neck","v-neck","v neck",
        "sweetheart","sweetheart neckline","halter","boat neck",
        "square neck","scoop neck","off shoulder","off-shoulder",
        "one shoulder","one-shoulder","high neck","turtleneck"
    }

    GARMENTS = {
        "shirt","t-shirt","tee","top","blouse","tank","dress","gown",
        "skirt","jean","pant","trouser","legging","short",
        "hoodie","coat","raincoat","jacket","blazer","cardigan","sweater",
        "kurta","saree","jumpsuit","romper","vest","suit"
    }

    HAIR_COLORS = {
        "black","brown","blonde","blond","grey","gray",
        "red","auburn","silver","white"
    }

    # base/lemma forms — "standing"/"stood"/"stands" all lemmatize to "stand"
    POSES = {
        "stand","walk","run",
        "sit","kneel","lean",
        "pose","look","face"
    }

    HAIR_STYLES = {
        "curly","straight","wavy","loose waves",
        "ponytail","bun","braid","bob"
    }

    FOOTWEAR = {
        "shoe","heel","boot",
        "sandal","sneaker","loafer","flat"
    }

    ACCESSORIES = {
        "bag","handbag","backpack","belt","watch",
        "hat","cap","necklace","bracelet","ring",
        "earring","scarf","glasses","sunglasses","tie"
    }

    PATTERNS = {
        "plain","solid","striped","checked","checkered",
        "plaid","floral","printed","graphic",
        "paisley","polka","animal","camouflage"
    }

    MATERIALS = {
        "cotton","denim","leather","silk","linen","lace",
        "knit","velvet","satin","polyester",
        "wool","chiffon","mesh"
    }

    SLEEVES = {
        "sleeveless","short-sleeve","long-sleeve",
        "half-sleeve","full-sleeve"
    }

    FITS = {
        "oversized","slim","loose","tight",
        "regular","relaxed","fitted"
    }

    LENGTHS = {
        "mini","midi","maxi","cropped","long"
    }

    # base/lemma forms — "standing outdoors"/"outdoor"/"outdoors" all normalize
    SETTINGS = {
        "indoor","outdoor","runway","fashion","street",
        "park","beach","studio","office","mall",
        "restaurant","home","stage","party","wedding","city"
    }

    ENVIRONMENT = {
        "tree","forest","grass","leaf",
        "flower","river","lake","water",
        "mountain","snow","road","street","beach",
        "park","building","sky","cloud","bush"
    }

    VIBES = {
        "casual","formal","sporty","business","professional",
        "vintage","elegant","party",
        "minimalist","bohemian","luxury",
        "streetwear"
    }

    # base/lemma forms — "smiling"/"smiled"/"smiles" all lemmatize to "smile"
    EXPRESSIONS = {
        "smile","laugh","happy","serious",
        "thoughtful","neutral","confident",
        "sad","angry","surprised"
    }

    LIGHTING = {
        "bright","soft","warm","natural",
        "dim","dramatic","sunlit","sunlight"
    }

    OBJECTS = {
        "tree","flower",
        "leaf","road","street",
        "building","camera","bench",
        "chair","table","river","lake",
        "water","person",
        "car","grass","sky","cloud"
    }

    # --------------------------------------------------
    # Output Dictionary
    # --------------------------------------------------

    features = {

        # Primary fashion attributes
        "gender": "",
        "age_group": "",

        "garments": [],
        "colors": [],
        "patterns": [],
        "materials": [],
        "necklines": [],
        "sleeves": [],
        "fit": [],
        "length": [],

        "footwear": [],
        "accessories": [],

        "hair_colors": [],
        "hair_styles": [],

        "expressions": [],
        "poses": [],

        # Scene information
        "setting": [],
        "environment": [],
        "objects": [],
        "lighting": [],
        "vibe": [],

        # Search keywords
        "keywords": [],

        # Per-garment bindings, e.g. a "red shirt and black pants" caption
        # produces [{"item": "shirt", "colors": ["red"], ...},
        #           {"item": "pant",  "colors": ["black"], ...}]
        # instead of flattening everything into one shared colors/garments bag.
        "garment_details": [],

        # NLP metadata (least important)
        "nouns": [],
        "adjectives": [],
        "verbs": [],
        "noun_chunks": []
    }

    # --------------------------------------------------
    # Token Extraction (checks BOTH lemma and raw text against
    # each vocab set, so inflected/tense variants aren't missed)
    # --------------------------------------------------

    for token in doc:

        lemma = token.lemma_.lower()
        raw = token.text.lower()
        forms = {lemma, raw}

        if token.pos_ in ["NOUN", "PROPN"]:
            features["nouns"].append(lemma)

        if token.pos_ == "VERB":
            features["verbs"].append(lemma)

        if token.pos_ == "ADJ":
            features["adjectives"].append(lemma)

        def add_if_match(vocab, bucket):
            hit = forms & vocab
            if hit:
                features[bucket].append(sorted(hit)[0])

        add_if_match(COLORS, "colors")
        add_if_match(GARMENTS, "garments")
        add_if_match(HAIR_COLORS, "hair_colors")
        add_if_match(HAIR_STYLES, "hair_styles")
        add_if_match(POSES, "poses")
        add_if_match(ENVIRONMENT, "environment")
        add_if_match(FOOTWEAR, "footwear")
        add_if_match(ACCESSORIES, "accessories")
        add_if_match(PATTERNS, "patterns")
        add_if_match(MATERIALS, "materials")
        add_if_match(SLEEVES, "sleeves")
        add_if_match(FITS, "fit")
        add_if_match(LENGTHS, "length")
        add_if_match(SETTINGS, "setting")
        add_if_match(VIBES, "vibe")
        add_if_match(EXPRESSIONS, "expressions")
        add_if_match(LIGHTING, "lighting")
        add_if_match(OBJECTS, "objects")

    # --------------------------------------------------
    # Multi-word Attribute Matching (genuinely multi-token terms only —
    # single-word terms are already covered by the lemma/text check above)
    # --------------------------------------------------

    for neckline in NECKLINES:
        if neckline in text:
            features["necklines"].append(neckline)

    for style in HAIR_STYLES:
        if " " in style and style in text:
            features["hair_styles"].append(style)

    # --------------------------------------------------
    # Gender
    # --------------------------------------------------

    if any(x in text for x in ["woman","female","girl","lady"," she "," her "]):
        features["gender"] = "female"

    elif any(x in text for x in ["man","male","boy","gentleman"," he "," his "]):
        features["gender"] = "male"

    # --------------------------------------------------
    # Age Group
    # --------------------------------------------------

    if any(x in text for x in ["child","kid","toddler"]):
        features["age_group"] = "child"

    elif any(x in text for x in ["teen","teenager"]):
        features["age_group"] = "teen"

    elif any(x in text for x in ["young","young adult"]):
        features["age_group"] = "young"

    elif "adult" in text:
        features["age_group"] = "adult"

    elif any(x in text for x in ["elderly","senior","old man","old woman"]):
        features["age_group"] = "elderly"

    # --------------------------------------------------
    # Per-Garment Attribute Binding
    #
    # For every token that IS a garment/footwear noun, walk its direct
    # dependency children (adjectival modifiers, compounds) to find the
    # color/pattern/material/fit/sleeve/length words that specifically
    # describe THAT garment — not just "any color mentioned anywhere in
    # the caption". This is what lets "red shirt and black pants" match
    # a query for "red shirt" without also matching "red pants".
    # --------------------------------------------------

    garment_vocab = GARMENTS | FOOTWEAR | ACCESSORIES
    seen_tokens = set()

    for token in doc:
        lemma = token.lemma_.lower()
        raw = token.text.lower()

        if not ((lemma in garment_vocab or raw in garment_vocab)):
            continue

        if token.i in seen_tokens:
            continue
        seen_tokens.add(token.i)

        item_colors, item_patterns, item_materials = [], [], []
        item_fit, item_sleeves, item_length = [], [], []

        # Modifiers directly attached to THIS garment token only.
        # (Deliberately not walking up to token.head's other children —
        # in "red shirt and black pants" the two garments are siblings
        # joined by "conj", and pulling in the head's children would
        # leak "red" from shirt onto pants. Direct children already
        # catch amod/compound modifiers correctly regardless of
        # whether this garment is the conjunction root or a conjunct.)
        for child in token.children:
            cforms = {child.lemma_.lower(), child.text.lower()}

            if child.dep_ not in ("amod", "compound", "conj", "nmod"):
                continue

            if cforms & COLORS:
                item_colors.append(sorted(cforms & COLORS)[0])
            if cforms & PATTERNS:
                item_patterns.append(sorted(cforms & PATTERNS)[0])
            if cforms & MATERIALS:
                item_materials.append(sorted(cforms & MATERIALS)[0])
            if cforms & FITS:
                item_fit.append(sorted(cforms & FITS)[0])
            if cforms & SLEEVES:
                item_sleeves.append(sorted(cforms & SLEEVES)[0])
            if cforms & LENGTHS:
                item_length.append(sorted(cforms & LENGTHS)[0])

            # handle "red and black shirt" — conjuncts of an amod
            for cc in child.children:
                if cc.dep_ == "conj":
                    ccforms = {cc.lemma_.lower(), cc.text.lower()}
                    if ccforms & COLORS:
                        item_colors.append(sorted(ccforms & COLORS)[0])
                    if ccforms & PATTERNS:
                        item_patterns.append(sorted(ccforms & PATTERNS)[0])
                    if ccforms & MATERIALS:
                        item_materials.append(sorted(ccforms & MATERIALS)[0])

        features["garment_details"].append({
            "item": lemma if lemma in garment_vocab else raw,
            "colors": sorted(set(item_colors)),
            "patterns": sorted(set(item_patterns)),
            "materials": sorted(set(item_materials)),
            "fit": sorted(set(item_fit)),
            "sleeves": sorted(set(item_sleeves)),
            "length": sorted(set(item_length)),
        })

    # --------------------------------------------------
    # Noun Chunks
    # --------------------------------------------------

    features["noun_chunks"] = [chunk.text.lower() for chunk in doc.noun_chunks]

    # --------------------------------------------------
    # Remove Duplicates
    # --------------------------------------------------

    for key in features:
        if isinstance(features[key], list) and key != "garment_details":
            features[key] = sorted(set(features[key]))

    # --------------------------------------------------
    # Search Keywords
    # --------------------------------------------------

    features["keywords"] = sorted(set(
        features["colors"] +
        features["garments"] +
        features["necklines"] +
        features["hair_colors"] +
        features["hair_styles"] +
        features["poses"] +
        features["environment"] +
        features["footwear"] +
        features["accessories"] +
        features["patterns"] +
        features["materials"] +
        features["sleeves"] +
        features["fit"] +
        features["length"] +
        features["setting"] +
        features["vibe"] +
        features["expressions"] +
        features["lighting"] +
        features["objects"]
    ))

    return features

In [ ]:
# Fields we treat as "hard attribute constraints" when the query mentions them.
# These map 1:1 onto keys in both the query features and the indexed metadata.
# These are checked as a general "did the image get tagged with this at all"
# overlap — good recall, but NOT garment-aware (see garment_details below).
ATTRIBUTE_FIELDS = [
    "colors", "garments", "patterns", "materials", "necklines",
    "sleeves", "fit", "length", "footwear", "accessories",
    "hair_colors", "hair_styles", "poses", "setting",
    "environment", "vibe", "expressions", "lighting",
]

def parse_query(query):
    """Extract structured attributes the user is asking for from free text."""
    features = extract_spacy_features(query)

    requested = {
        field: set(features[field])
        for field in ATTRIBUTE_FIELDS
        if features[field]
    }

    if features["gender"]:
        requested["gender"] = {features["gender"]}
    if features["age_group"]:
        requested["age_group"] = {features["age_group"]}

    # Per-garment bindings, e.g. "red shirt and black pants" ->
    # [{"item": "shirt", "colors": ["red"], ...}, {"item": "pant", "colors": ["black"], ...}]
    # This is what lets the retriever tell "red shirt" apart from "red pants".
    garment_details = features["garment_details"]

    return requested, garment_details


# Quick check
parse_query("red shirt and black pants")

## Section 4 — Hybrid Search Engine

`search(query, k=9, alpha=0.5, garment_weight=0.6)`:

1. Encode `query` with CLIP → search FAISS for the top `k * oversample` visually similar candidates.
2. Parse `query` into:
   - flat requested attributes (`parse_query`'s `requested` dict) — a general "does the image have this tag anywhere" signal.
   - per-garment bindings (`garment_details`) — e.g. "red shirt and black pants" becomes two separate garment+color pairs, not one shared bag of colors and one shared bag of garments.
3. Score each candidate:
   - `flat_score` = fraction of requested attribute fields satisfied anywhere in the image's metadata.
   - `garment_score` = fraction of requested garment+attribute pairs that are satisfied by the *same* garment in the image (a "red shirt" request only counts if some garment tagged "shirt" on that image is also tagged "red" — matching red on the pants doesn't count).
   - `attribute_score = garment_weight * garment_score + (1 - garment_weight) * flat_score` when the query names specific garments; otherwise `attribute_score = flat_score`.
   - `final_score = alpha * clip_similarity + (1 - alpha) * attribute_score`.
4. Return the top `k` by combined score.

`alpha` trades off visual similarity vs. attributes overall. `garment_weight` (only relevant when the query names garments) trades off strict per-garment binding vs. general "these colors/garments appear somewhere" matching.

In [ ]:
def attribute_match_ratio(requested, record):
    """
    Fraction of requested attribute fields satisfied by this record's metadata.
    A field is "satisfied" if there's any overlap between what was asked for
    and what the record has tagged for that field, ANYWHERE on the image —
    this does not check that a color and a garment belong to the same item.
    """
    if not requested:
        return 1.0

    hits = 0
    for field, wanted_values in requested.items():

        if field in ("gender", "age_group"):
            record_value = record.get(field, "")
            if record_value and record_value in wanted_values:
                hits += 1
            continue

        record_values = set(record.get(field, []))
        if wanted_values & record_values:
            hits += 1

    return hits / len(requested)


def garment_match_ratio(query_garment_details, record_garment_details):
    """
    Fraction of the query's requested garment+attribute bundles that are
    satisfied by the SAME garment on the image (not just anywhere on it).

    e.g. query garment_details = [{"item": "shirt", "colors": ["red"], ...}]
    only scores a hit if the record has a garment tagged "shirt" whose own
    colors include "red" — a red pair of pants elsewhere on the same image
    does not count.
    """
    if not query_garment_details:
        return 1.0

    hits = 0
    for wanted in query_garment_details:
        wanted_item = wanted["item"]
        wanted_colors = set(wanted.get("colors", []))
        wanted_patterns = set(wanted.get("patterns", []))
        wanted_materials = set(wanted.get("materials", []))

        matched = False
        for candidate in record_garment_details:
            if candidate.get("item") != wanted_item:
                continue

            candidate_colors = set(candidate.get("colors", []))
            candidate_patterns = set(candidate.get("patterns", []))
            candidate_materials = set(candidate.get("materials", []))

            # item matches; now every attribute the query specified for
            # this garment must also be present on this specific garment
            ok = True
            if wanted_colors and not (wanted_colors & candidate_colors):
                ok = False
            if wanted_patterns and not (wanted_patterns & candidate_patterns):
                ok = False
            if wanted_materials and not (wanted_materials & candidate_materials):
                ok = False

            if ok:
                matched = True
                break

        if matched:
            hits += 1

    return hits / len(query_garment_details)

In [ ]:
def search(query, k=9, alpha=0.5, garment_weight=0.6, oversample=8):
    """
    Natural language search over the fashion image index.

    query           : free text, e.g. "woman in a red shirt and black pants"
    k               : number of results to return
    alpha           : weight on CLIP visual similarity vs. attributes overall (0-1)
    garment_weight  : when the query names specific garments, how much to trust
                       strict per-garment attribute binding vs. general flat
                       attribute overlap (0-1). Higher = stricter about which
                       color belongs to which garment.
    oversample      : how many extra CLIP candidates to pull before re-ranking,
                       as a multiple of k (wider net = better recall for rare
                       attribute combos)
    """

    # ---- 1. CLIP semantic candidates ----
    with torch.no_grad():
        text_tokens = clip_tokenizer([query]).to(device)
        text_features = clip_model.encode_text(text_tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        text_features = text_features.cpu().numpy().astype("float32")

    n_candidates = min(k * oversample, faiss_index.ntotal)
    clip_scores, candidate_idx = faiss_index.search(text_features, n_candidates)
    clip_scores = clip_scores[0]
    candidate_idx = candidate_idx[0]

    # ---- 2. Parse query attributes ----
    requested, query_garment_details = parse_query(query)

    # ---- 3. Score candidates ----
    scored = []
    for clip_score, idx in zip(clip_scores, candidate_idx):
        if idx == -1:
            continue

        record = metadata_df.iloc[idx].to_dict()

        flat_score = attribute_match_ratio(requested, record)

        if query_garment_details:
            record_garment_details = record.get("garment_details", [])
            garment_score = garment_match_ratio(query_garment_details, record_garment_details)
            attr_score = garment_weight * garment_score + (1 - garment_weight) * flat_score
        else:
            garment_score = None
            attr_score = flat_score

        # CLIP cosine similarity is roughly in [-1, 1]; rescale to [0, 1]
        clip_score_norm = (clip_score + 1) / 2

        final_score = alpha * clip_score_norm + (1 - alpha) * attr_score

        scored.append({
            "image_id": record["image_id"],
            "image_path": record["image_path"],
            "description": record.get("description", ""),
            "clip_score": float(clip_score),
            "attribute_match": attr_score,
            "garment_match": garment_score,
            "final_score": final_score,
            "metadata": record,
        })

    scored.sort(key=lambda r: r["final_score"], reverse=True)

    return scored[:k], requested

## Section 5 — Display Results

In [ ]:
def show_results(query, results, requested):

    print(f"Query: \"{query}\"")
    if requested:
        print("Detected attributes:", {k: sorted(v) for k, v in requested.items()})
    else:
        print("Detected attributes: none (pure visual search)")
    print()

    n = len(results)
    cols = min(3, n) if n > 0 else 1
    rows = (n + cols - 1) // cols

    plt.figure(figsize=(5 * cols, 6 * rows))

    for i, r in enumerate(results):
        image = Image.open(r["image_path"]).convert("RGB")

        plt.subplot(rows, cols, i + 1)
        plt.imshow(image)
        plt.axis("off")

        garment_str = (
            f"  garment={r['garment_match']:.2f}"
            if r.get("garment_match") is not None else ""
        )
        plt.title(
            f"{r['image_id']}\n"
            f"clip={r['clip_score']:.3f}  attr={r['attribute_match']:.2f}"
            f"{garment_str}  score={r['final_score']:.3f}",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

## Section 6 — Example Queries

In [ ]:
# Try your own query
my_query = "A woman wearing a floral dress with strapless sweetheart neckline "
results, requested = search(my_query, k=9, alpha=0.6)
show_results(my_query, results, requested)

In [ ]:
results, requested = search("casual outfit with sneakers in the street", k=6)
show_results("casual outfit with sneakers in the street", results, requested)

In [ ]:
results, requested = search("woman wearing a black blazer over a white shirt", k=6)
show_results("woman wearing a black blazer over a white shirt", results, requested)

In [ ]:
# Try your own query
my_query = "A person in a bright yellow raincoat"
results, requested = search(my_query, k=12, alpha=0.6)
show_results(my_query, results, requested)

In [ ]:
# Try your own query
my_query = "Professional business attire inside a modern office."
results, requested = search(my_query, k=9, alpha=0.6)
show_results(my_query, results, requested)

In [ ]:
# Try your own query
my_query = "Someone wearing a blue shirt sitting on a park bench."
results, requested = search(my_query, k=3, alpha=0.6)
show_results(my_query, results, requested)

In [ ]:
# Try your own query
my_query = "Casual weekend outfit for a city walk."
results, requested = search(my_query, k=9, alpha=0.6)
show_results(my_query, results, requested)

In [ ]:
# Try your own query
my_query = "A red tie and a white shirt in a formal setting."
results, requested = search(my_query, k=9, alpha=0.6)
show_results(my_query, results, requested)

In [ ]:
# Garment-color binding check: this should now favor images where the
# SHIRT is red and the PANTS are black, not just any image containing
# both colors somewhere.
results, requested = search("red shirt and black pants", k=6)
show_results("red shirt and black pants", results, requested)